## Stage 03 — Campaign Quality Survey

Navigate all survey files one at a time and mark each as `good`, `uncertain`, or `bad`.
Decisions are saved to `quality_manifest.yaml` and consumed by `03a_align_wyo.ipynb`
and `03b_align_mml.ipynb` to pre-reject bad files before the alignment widget.

**Workflow:**
1. Run all cells through the widget
2. Use Prev / Next to step through files; look at the plot
3. Click Good / Uncertain / Bad; add a note if helpful
4. Manifest saves after every click — safe to close and re-open at any time

**Status meanings:**
- `good` — clean session; lag correction should be reliable
- `uncertain` — present but noisy, short, or questionable; alignment widget will show `[?]`
- `bad` — unusable; alignment widget auto-skips these files (Commit there to override)

**Manifest structure:** keyed by `raw_stem` (session key shared by Raw, Eng, and Spectra
from the same recording), so one decision covers all file types for a session.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
import yaml
from IPython.display import display, HTML, clear_output

sys.path.insert(0, str(Path().resolve().parent))
from paths import STAGE_02_DIR, STAGE_03_DIR, QUALITY_MANIFEST_PATH
from src.align import raw_stem, date_tag

display(HTML('<style>.plotly-graph-div { width: 100% !important; }</style>'))
print('Imports OK')

In [ ]:
# Instruments to survey: {name: (parquet_dir, [cols_to_plot])}
# Cols are plotted as separate traces on the same axis.
# CH4_ppm is the primary quality signal for gas instruments.
# H2O_ppm added for LANL gas instruments (inlet behaviour + MML alignment signal).
# u_ms used for Anem (wind present and sensible = instrument working).
# speed_ms used for GPS (shows vehicle movement and GPS lock quality).
SURVEY_INSTRUMENTS = {
    'WYO_picarro':        (STAGE_02_DIR / 'WYO_picarro',                ['CH4_ppm']),
    'WYO_aerisultra460':  (STAGE_02_DIR / 'WYO_aerisultra460' / 'Raw',  ['CH4_ppm']),
    'LANL_aerisultra321': (STAGE_02_DIR / 'LANL_aerisultra321' / 'Raw', ['CH4_ppm', 'H2O_ppm']),
    'LANL_aerispico017':  (STAGE_02_DIR / 'LANL_aerispico017'  / 'Raw', ['CH4_ppm', 'H2O_ppm']),
    'UOU_LGR':            (STAGE_02_DIR / 'UOU_LGR',                    ['CH4_ppm']),
    'LANL_Anem':          (STAGE_02_DIR / 'LANL_Anem',                  ['u_ms']),
    'LANL_GPS':           (STAGE_02_DIR / 'LANL_GPS',                   ['speed_ms']),
}

# Instruments that are trusted references — surveyed but flagged in the title
REFERENCE_INSTRUMENTS = {'WYO_picarro'}

# Colors per instrument (consistent with 03a/03b)
INST_COLORS = {
    'WYO_picarro':        '#2E86AB',
    'WYO_aerisultra460':  '#E67E22',
    'LANL_aerisultra321': '#27AE60',
    'LANL_aerispico017':  '#8E44AD',
    'UOU_LGR':            '#E74C3C',
    'LANL_Anem':          '#16A085',
    'LANL_GPS':           '#7F8C8D',
}
COL_COLORS = [
    '#2980B9', '#E67E22', '#27AE60', '#8E44AD', '#E74C3C',
    '#16A085', '#F39C12', '#2C3E50', '#D35400', '#1ABC9C',
    '#C0392B', '#7F8C8D', '#3498DB', '#2ECC71', '#9B59B6',
]  # per-column within a file

print('Config OK')

In [ ]:
def _load_manifest():
    if QUALITY_MANIFEST_PATH.exists():
        with open(QUALITY_MANIFEST_PATH) as fh:
            return yaml.safe_load(fh) or {}
    return {}

def _save_manifest(manifest):
    STAGE_03_DIR.mkdir(parents=True, exist_ok=True)
    with open(QUALITY_MANIFEST_PATH, 'w') as fh:
        yaml.dump(manifest, fh, default_flow_style=False, sort_keys=True)

print('Helpers loaded.')

In [ ]:
# Build the full ordered file list: [(instrument, path), ...]
# For each instrument, collect:
#   1. Files directly in the instrument dir (UTC-corrected or trusted timestamps)
#   2. Files in no_coverage/ subdir (LANL instruments where no logger match was found;
#      timestamps are Mountain Time, not UTC — pipeline routes these to bad_timestamp/
#      automatically, but you can still note whether the data itself is usable)
survey_entries = []
for inst, (inst_dir, cols) in SURVEY_INSTRUMENTS.items():
    if not inst_dir.exists():
        print(f'  [SKIP]  {inst} — {inst_dir} not found')
        continue
    main_files = sorted(inst_dir.glob('*.parquet'))
    nc_files   = sorted((inst_dir / 'no_coverage').glob('*.parquet')) \
                 if (inst_dir / 'no_coverage').exists() else []
    for f in main_files:
        survey_entries.append((inst, f))
    for f in nc_files:
        survey_entries.append((inst, f))
    nc_str = f' + {len(nc_files)} no_coverage' if nc_files else ''
    print(f'  {inst:<30}  {len(main_files):>4} files{nc_str}')

print(f'\nTotal: {len(survey_entries)} files to survey')

In [ ]:
import sys as _sys
import threading as _threading

if not hasattr(_sys, '_survey_lock'):
    _sys._survey_lock = _threading.Lock()

manifest = _load_manifest()
n_existing = sum(len(v) for v in manifest.values())
print(f'Manifest: {n_existing} existing entries  ({QUALITY_MANIFEST_PATH})')

if not survey_entries:
    print('No files found — check SURVEY_INSTRUMENTS paths.')
else:
    for _wname in ['reason_txt', 'inst_drop']:
        _w = globals().get(_wname)
        if _w is not None:
            try: _w.unobserve_all()
            except Exception: pass
    for _wname in ['_sv_root', 'sv_fig', 'sv_log', 'inst_drop',
                   'btn_prev', 'btn_next', 'reason_txt', 'sv_title', 'status_lbl',
                   'inst_progress', 'btn_set_good', 'btn_set_uncertain', 'btn_set_bad',
                   'btn_mark_good', 'btn_mark_uncertain', 'btn_mark_bad', 'btn_mark_unreview',
                   'col_selector_box']:
        _w = globals().get(_wname)
        if _w is not None:
            try: _w.close()
            except Exception: pass

    inst_order      = [i for i in SURVEY_INSTRUMENTS if any(i == inst for inst, _ in survey_entries)]
    entries_by_inst = {inst: [f for i, f in survey_entries if i == inst] for inst in inst_order}

    sv_state      = {'inst': inst_order[0], 'file_idx': 0, 'navigating': False}
    STATUS_COLORS = {'good': '#1E8449', 'uncertain': '#D35400', 'bad': '#C0392B'}
    _col_cache    = {}

    def _get_plot_cols(inst):
        if inst in _col_cache:
            return _col_cache[inst]
        files = entries_by_inst[inst]
        if not files:
            return list(SURVEY_INSTRUMENTS[inst][1])
        try:
            import pyarrow.parquet as pq
            schema = pq.read_schema(str(files[0]))
            skip   = {'ts_status', 'lag_ref'}
            cols   = [n for n in schema.names if n not in skip]
        except Exception:
            cols = list(SURVEY_INSTRUMENTS[inst][1])
        _col_cache[inst] = cols
        return cols

    MAX_TRACES = 20
    sv_fig = go.FigureWidget(layout=go.Layout(
        height=360, autosize=True, margin=dict(l=55, r=10, t=30, b=30),
        yaxis=dict(tickfont=dict(size=10)),
        legend=dict(x=1.01, y=1, xanchor='left', font=dict(size=9)),
        hovermode='x unified',
    ))
    for ci in range(MAX_TRACES):
        sv_fig.add_scattergl(name='', mode='lines', visible=False,
                             line=dict(color=COL_COLORS[ci % len(COL_COLORS)], width=1.2))

    sv_title         = widgets.HTML(value='')
    sv_log           = widgets.HTML(value='', layout=widgets.Layout(min_height='24px', padding='3px 6px'))
    inst_progress    = widgets.HTML(value='')
    col_selector_box = widgets.HBox([], layout=widgets.Layout(flex_flow='row wrap', gap='4px 8px', margin='2px 0 4px 0'))
    inst_drop        = widgets.Dropdown(
        options=inst_order, value=inst_order[0],
        layout=widgets.Layout(width='220px'))
    btn_prev = widgets.Button(description='◀ Prev', layout=widgets.Layout(width='88px'))
    btn_next = widgets.Button(description='Next ▶', layout=widgets.Layout(width='88px'))

    # Per-file status buttons — prominent, full-width, color reflects active state
    btn_set_good      = widgets.Button(description='✓  Good',      layout=widgets.Layout(width='100%', height='34px'))
    btn_set_uncertain = widgets.Button(description='~  Uncertain',  layout=widgets.Layout(width='100%', height='34px'))
    btn_set_bad       = widgets.Button(description='✗  Bad',        layout=widgets.Layout(width='100%', height='34px'))
    status_lbl        = widgets.HTML(value='')
    reason_txt        = widgets.Text(placeholder='note (optional)', layout=widgets.Layout(width='220px'))

    # Bulk buttons — small, plain (no button_style), clearly secondary
    btn_mark_good      = widgets.Button(description='all good',       layout=widgets.Layout(width='100%', height='26px'))
    btn_mark_uncertain = widgets.Button(description='all uncertain',  layout=widgets.Layout(width='100%', height='26px'))
    btn_mark_bad       = widgets.Button(description='all bad',        layout=widgets.Layout(width='100%', height='26px'))
    btn_mark_unreview  = widgets.Button(description='all unreviewed', layout=widgets.Layout(width='100%', height='26px'))

    def _update_file_btns(status):
        btn_set_good.button_style      = 'success' if status == 'good'      else ''
        btn_set_uncertain.button_style = 'warning' if status == 'uncertain' else ''
        btn_set_bad.button_style       = 'danger'  if status == 'bad'       else ''

    def _build_col_checkboxes(inst):
        available = _get_plot_cols(inst)
        defaults  = set(SURVEY_INSTRUMENTS[inst][1])
        boxes = []
        for col in available:
            cb = widgets.Checkbox(
                value=(col in defaults),
                description=col,
                indent=False,
                layout=widgets.Layout(width='auto'),
                style={'description_width': 'initial'},
            )
            cb.observe(_on_col_change, names='value')
            boxes.append(cb)
        col_selector_box.children = tuple(boxes)

    def _selected_cols(inst):
        cols = [cb.description for cb in col_selector_box.children if cb.value]
        return cols if cols else list(SURVEY_INSTRUMENTS[inst][1])

    def _progress_html():
        current = _load_manifest()
        parts = []
        for inst in inst_order:
            files = entries_by_inst[inst]
            n     = len(files)
            ient  = current.get(inst, {})
            g = sum(1 for f in files if ient.get(raw_stem(f), {}).get('status') == 'good')
            u = sum(1 for f in files if ient.get(raw_stem(f), {}).get('status') == 'uncertain')
            b = sum(1 for f in files if ient.get(raw_stem(f), {}).get('status') == 'bad')
            r    = g + u + b
            done = r == n
            active  = inst == sv_state['inst']
            color   = INST_COLORS.get(inst, '#000')
            weight  = 'bold' if active else 'normal'
            opacity = '1'    if active else '0.5'
            tick    = '&nbsp;✓' if done else ''
            parts.append(
                f'<span style="color:{color};font-weight:{weight};opacity:{opacity}">{inst}</span>'
                f'<small style="color:#888;opacity:{opacity}">&nbsp;{r}/{n}'
                f'&nbsp;G:{g}&nbsp;?:{u}&nbsp;X:{b}{tick}</small>'
            )
        return '&nbsp;&nbsp;|&nbsp;&nbsp;'.join(parts)

    def _refresh_progress():
        inst_progress.value = _progress_html()

    def _refresh_status_lbl(inst, stem):
        current = _load_manifest()
        status  = current.get(inst, {}).get(stem, {}).get('status', '')
        color   = STATUS_COLORS.get(status, '#888')
        label   = f'<b style="color:{color}">{status}</b>' if status else '<span style="color:#aaa">not reviewed</span>'
        status_lbl.value = f'<small>Saved: {label}</small>'

    def _navigate(inst, file_idx):
        if sv_state['navigating']: return
        sv_state['navigating'] = True
        try:
            files = entries_by_inst[inst]
            if not files:
                sv_title.value = f'<b>{inst}: no files found</b>'
                return
            file_idx = max(0, min(file_idx, len(files) - 1))
            sv_state['file_idx'] = file_idx
            f         = files[file_idx]
            stem      = raw_stem(f)
            is_nc     = f.parent.name == 'no_coverage'
            inst_cols = _selected_cols(inst)
            try:
                df = pd.read_parquet(f, columns=inst_cols)
            except Exception as e:
                sv_title.value = f'<span style="color:red">ERROR reading {f.name}: {e}</span>'
                return
            with sv_fig.batch_update():
                sv_fig.layout.yaxis.autorange = True
                sv_fig.layout.xaxis.autorange = True
                for ci, col in enumerate(inst_cols[:MAX_TRACES]):
                    s = df[col].dropna()
                    # Pass Python lists — numpy arrays trigger a plotly/ipywidgets bug
                    # in _remove_overlapping_props when the frontend syncs state back.
                    sv_fig.data[ci].x = s.index.tolist()
                    sv_fig.data[ci].y = s.values.tolist()
                    sv_fig.data[ci].name    = col
                    sv_fig.data[ci].visible = True
                for ci in range(min(len(inst_cols), MAX_TRACES), MAX_TRACES):
                    sv_fig.data[ci].x = []
                    sv_fig.data[ci].y = []
                    sv_fig.data[ci].visible = False
                sv_fig.layout.yaxis.title = inst_cols[0] if len(inst_cols) == 1 else 'value'
            first_col  = df[inst_cols[0]].dropna()
            t0_str     = first_col.index[0].strftime('%H:%M') if len(first_col) else '?'
            t1_str     = first_col.index[-1].strftime('%H:%M') if len(first_col) else '?'
            dtag       = date_tag(f)
            notes      = []
            if inst in REFERENCE_INSTRUMENTS: notes.append('reference instrument')
            if is_nc: notes.append('⚠ MT clock — no logger coverage')
            note_str   = ('  ·  ' + '  ·  '.join(notes)) if notes else ''
            ts_label   = 'MT' if is_nc else 'UTC'
            inst_color = INST_COLORS.get(inst, '#000')
            nc_bg      = ' style="background:#FFF3CD;padding:0 3px;border-radius:2px"' if is_nc else ''
            sv_title.value = (
                f'[{file_idx+1}/{len(files)}]  '
                f'<span style="color:{inst_color};font-weight:bold">{inst}</span>  '
                f'<span{nc_bg} style="color:#555">{f.name}</span>'
                f'<br><small style="color:#777">20{dtag[:2]}-{dtag[2:4]}-{dtag[4:6]}  '
                f'{t0_str}–{t1_str} {ts_label}  ·  {len(first_col):,} rows{note_str}</small>'
            )
            current = _load_manifest()
            reason_txt.value = current.get(inst, {}).get(stem, {}).get('reason', '')
            _update_file_btns(current.get(inst, {}).get(stem, {}).get('status', ''))
            _refresh_status_lbl(inst, stem)
        finally:
            sv_state['navigating'] = False

    def _on_col_change(change):
        if sv_state['navigating']: return
        _navigate(sv_state['inst'], sv_state['file_idx'])

    def _on_inst_change(change):
        if sv_state['navigating']: return
        new_inst = change['new']
        sv_state['inst']     = new_inst
        sv_state['file_idx'] = 0
        sv_log.value = ''
        _build_col_checkboxes(new_inst)
        _refresh_progress()
        _navigate(new_inst, 0)

    def _set_file_status(new_status):
        if sv_state['navigating']: return
        inst  = sv_state['inst']
        files = entries_by_inst[inst]
        if sv_state['file_idx'] >= len(files): return
        f    = files[sv_state['file_idx']]
        stem = raw_stem(f)
        with _sys._survey_lock:
            current = _load_manifest()
            if current.get(inst, {}).get(stem, {}).get('status') == new_status:
                return
            current.setdefault(inst, {})[stem] = {
                **current.get(inst, {}).get(stem, {}),
                'status': new_status,
            }
            _save_manifest(current)
        _update_file_btns(new_status)
        _refresh_status_lbl(inst, stem)
        _refresh_progress()
        sv_log.value = f'<small style="color:#555">&nbsp;&nbsp;{stem} → {new_status}</small>'

    def _on_reason(change):
        if sv_state['navigating']: return
        inst  = sv_state['inst']
        files = entries_by_inst[inst]
        if sv_state['file_idx'] >= len(files): return
        f    = files[sv_state['file_idx']]
        stem = raw_stem(f)
        with _sys._survey_lock:
            current = _load_manifest()
            current.setdefault(inst, {})[stem] = {
                **current.get(inst, {}).get(stem, {}),
                'reason': change['new'],
            }
            _save_manifest(current)

    def _mark_all(status):
        inst  = sv_state['inst']
        files = entries_by_inst[inst]
        with _sys._survey_lock:
            current = _load_manifest()
            for f in files:
                stem = raw_stem(f)
                current.setdefault(inst, {})[stem] = {
                    **current.get(inst, {}).get(stem, {}),
                    'status': status,
                }
            _save_manifest(current)
        _refresh_progress()
        if files:
            stem = raw_stem(files[sv_state['file_idx']])
            _update_file_btns(status)
            _refresh_status_lbl(inst, stem)
        sv_log.value = (
            f'<small style="color:#555">&nbsp;&nbsp;'
            f'Marked all {len(files)} <b>{inst}</b> files as <b>{status}</b></small>'
        )

    def _mark_all_unreviewed():
        inst  = sv_state['inst']
        files = entries_by_inst[inst]
        with _sys._survey_lock:
            current = _load_manifest()
            for f in files:
                stem  = raw_stem(f)
                entry = current.get(inst, {}).get(stem, {})
                entry.pop('status', None)
                if entry:
                    current.setdefault(inst, {})[stem] = entry
                elif stem in current.get(inst, {}):
                    del current[inst][stem]
            _save_manifest(current)
        _refresh_progress()
        if files:
            stem = raw_stem(files[sv_state['file_idx']])
            _update_file_btns('')
            _refresh_status_lbl(inst, stem)
        sv_log.value = (
            f'<small style="color:#555">&nbsp;&nbsp;'
            f'Cleared all {len(files)} <b>{inst}</b> statuses</small>'
        )

    inst_drop.observe(_on_inst_change, names='value')
    reason_txt.observe(_on_reason,     names='value')
    btn_set_good.on_click(     lambda _: _set_file_status('good'))
    btn_set_uncertain.on_click(lambda _: _set_file_status('uncertain'))
    btn_set_bad.on_click(      lambda _: _set_file_status('bad'))
    btn_mark_good.on_click(     lambda _: _mark_all('good'))
    btn_mark_uncertain.on_click(lambda _: _mark_all('uncertain'))
    btn_mark_bad.on_click(      lambda _: _mark_all('bad'))
    btn_mark_unreview.on_click( lambda _: _mark_all_unreviewed())

    def _on_prev(_):
        if sv_state['navigating']: return
        _navigate(sv_state['inst'], sv_state['file_idx'] - 1)
    def _on_next(_):
        if sv_state['navigating']: return
        _navigate(sv_state['inst'], sv_state['file_idx'] + 1)
    btn_prev.on_click(_on_prev)
    btn_next.on_click(_on_next)

    inst_row    = widgets.HBox([widgets.Label('Instrument:'), inst_drop],
                               layout=widgets.Layout(gap='6px', align_items='center', margin='4px 0'))
    nav_row     = widgets.HBox([btn_prev, btn_next],
                               layout=widgets.Layout(gap='6px', margin='3px 0'))
    file_status = widgets.VBox(
        [btn_set_good, btn_set_uncertain, btn_set_bad, status_lbl, reason_txt],
        layout=widgets.Layout(gap='4px', margin='4px 0'),
    )
    bulk_label  = widgets.HTML('<small style="color:#999;font-style:italic">bulk — current instrument:</small>')
    bulk_col    = widgets.VBox(
        [bulk_label, btn_mark_good, btn_mark_uncertain, btn_mark_bad, btn_mark_unreview],
        layout=widgets.Layout(gap='3px', margin='2px 0'),
    )
    _sep  = widgets.HTML('<hr style="margin:8px 0;border:none;border-top:1px solid #ddd">')
    _sep2 = widgets.HTML('<hr style="margin:8px 0;border:none;border-top:1px solid #ddd">')
    left_panel  = widgets.VBox(
        [inst_row, nav_row, _sep, file_status, _sep2, bulk_col, sv_log],
        layout=widgets.Layout(width='265px', min_width='265px', padding='4px 14px 4px 4px'),
    )
    right_panel = widgets.VBox(
        [sv_title, col_selector_box, sv_fig, inst_progress],
        layout=widgets.Layout(flex='1', min_width='0', width='100%'),
    )
    _sv_root = widgets.HBox(
        [left_panel, right_panel],
        layout=widgets.Layout(width='100%', align_items='flex-start'),
    )
    _build_col_checkboxes(inst_order[0])
    _refresh_progress()
    clear_output(wait=True)
    display(_sv_root)
    _navigate(inst_order[0], 0)


---
## Summary

Run to see current manifest state.

In [ ]:
manifest = _load_manifest()
if not manifest:
    print('Manifest is empty.')
else:
    STATUS_CHAR = {'good': 'G', 'uncertain': '?', 'bad': 'X'}
    counts = {'good': 0, 'uncertain': 0, 'bad': 0, 'unreviewed': 0}
    for inst, entries in sorted(manifest.items()):
        if not entries: continue
        print(f'\n{inst}:')
        for stem, entry in sorted(entries.items()):
            status = entry.get('status', '')
            reason = entry.get('reason', '')
            char   = STATUS_CHAR.get(status, '-')
            note   = f'  # {reason}' if reason else ''
            print(f'  [{char}]  {stem}{note}')
            counts[status if status in counts else 'unreviewed'] += 1
    total = sum(counts.values())
    print(f'\nTotal: {total}  |  G={counts["good"]}  ?={counts["uncertain"]}  X={counts["bad"]}  -={counts["unreviewed"]}')
    print(f'Saved at: {QUALITY_MANIFEST_PATH}')